In [0]:
dbutils.widgets.text("catalog", "ridepulse", "Catalog name")
dbutils.widgets.text("source_path", "/Volumes/ridepulse/landing/raw/hvfhv/fhvhv_tripdata_2026-01.parquet", "Source parquet path")

catalog = dbutils.widgets.get("catalog")
source_path = dbutils.widgets.get("source_path")

bronze_table = f"{catalog}.bronze.trips"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.landing")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.lakeflow_demo")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.checkpoints")

from pyspark.sql.functions import current_timestamp, col

raw = spark.read.parquet(source_path)

tagged = raw.withColumn("ingestion_timestamp", current_timestamp()).withColumn(
    "source_file", col("_metadata.file_path")
)


table_exists = spark.catalog.tableExists(bronze_table)

if table_exists:
    already_ingested = [
        row.source_file
        for row in spark.table(bronze_table).select("source_file").distinct().collect()
    ]
    to_write = tagged.filter(~tagged["source_file"].isin(already_ingested))
else:
    to_write = tagged

new_row_count = to_write.count()
print(f"{new_row_count:,} new rows to append (already-ingested files were skipped)")


if new_row_count > 0:
    to_write.write.format("delta").mode("append").saveAsTable(bronze_table)
else:
    print("Nothing new — skipping write.")


total = spark.table(bronze_table).count()
print(f"{bronze_table}: {total:,} rows total")
display(spark.table(bronze_table).limit(10))
